# Week 10: MLOps, System Evaluation & Explainable AI
## Notebook 12: Production ML Telemetry: Prometheus Metrics & Latency Histograms

### 1. Conceptual Intuition & Theoretical Background
Serving ML models in production requires monitoring both **system health** (latency, throughput, memory) and **model health** (prediction distributions, feature drift).

**Prometheus Core Metric Types:**
1. **Counter:** Monotonically increasing value (e.g., total predictions, error count):
$$\text{Rate} = \frac{\Delta \text{Counter}}{\Delta t}$$
2. **Gauge:** Value that can fluctuate up and down (e.g., active worker count, VRAM utilization).
3. **Histogram:** Samples observations into discrete pre-configured buckets (e.g. latency buckets: $\le 10\text{ms}, \le 25\text{ms}, \le 50\text{ms}$):
Enables calculating rolling **P95 and P99 percentiles** without storing raw observation arrays:
$$\text{Histogram Quantile}(\phi) = \text{Interpolate}(\text{Buckets})$$

---

### 2. Scratch Python Implementation: Prometheus Telemetry Collector

In [ ]:
import numpy as np

class PrometheusMetricCollectorScratch:
    def __init__(self, latency_buckets=[10.0, 25.0, 50.0, 100.0, 250.0]):
        self.buckets = sorted(latency_buckets)
        self.bucket_counts = {b: 0 for b in self.buckets}
        self.bucket_counts[float('inf')] = 0
        
        # Counters
        self.total_requests = 0
        self.prediction_classes = {0: 0, 1: 0}
        self.total_latency_sum = 0.0
        
    def observe_request(self, latency_ms, predicted_class):
        self.total_requests += 1
        self.total_latency_sum += latency_ms
        self.prediction_classes[predicted_class] = self.prediction_classes.get(predicted_class, 0) + 1
        
        # Increment histogram buckets
        for b in self.buckets:
            if latency_ms <= b:
                self.bucket_counts[b] += 1
        self.bucket_counts[float('inf')] += 1
        
    def estimate_p95_latency(self):
        # Linear interpolation across histogram buckets
        target_count = 0.95 * self.total_requests
        prev_b = 0.0
        prev_count = 0
        
        for b in self.buckets:
            count = self.bucket_counts[b]
            if count >= target_count:
                # Interpolate within bucket
                fraction = (target_count - prev_count) / max(1, count - prev_count)
                return prev_b + fraction * (b - prev_b)
            prev_b = b
            prev_count = count
            
        return self.buckets[-1]

# Stream Simulated Production Requests
collector = PrometheusMetricCollectorScratch()
np.random.seed(42)

for _ in range(500):
    lat = float(np.random.exponential(scale=20.0) + 5.0)
    cls = int(np.random.binomial(1, 0.3))
    collector.observe_request(lat, cls)

p95_est = collector.estimate_p95_latency()

print("--- Prometheus Real-Time Metrics Exporter ---")
print(f"Total Requests Handled: {collector.total_requests}")
print(f"Mean Request Latency:   {collector.total_latency_sum / collector.total_requests:.2f} ms")
print(f"Estimated P95 Latency:  {p95_est:.2f} ms")
print(f"Class Distribution:     {collector.prediction_classes}")
print("\nHistogram Bucket Accumulation:")
for b in collector.buckets:
    print(f"  le_{b}ms: {collector.bucket_counts[b]}")

### 3. Step-by-Step Code Walkthrough
1. **Histogram Bucket Accumulation:** Categorizes latencies into bounded slots without keeping raw history in RAM.
2. **PromQL Quantile Approximation:** Interpolates percentile bounds from bucket boundaries.
3. **Class Ratio Monitoring:** Tracks output distributions to detect model drift in real time.

---

### 4. Industry Applications & Trade-offs
- **Kubernetes Model Pod Monitoring:** Alerting via Alertmanager when P99 latency breaches 200ms for more than 5 minutes.
- **Trade-off:** Poorly chosen histogram bucket boundaries yield inaccurate percentile estimates.

---

### 5. Top Interview Questions & Answers
**Q1: Why does Prometheus use Histograms rather than exact sorting for tracking latencies?**
> **Answer:** Computing exact percentiles across millions of requests requires storing and sorting every request latency, causing memory bloat. Histograms use fixed bucket counters to estimate percentiles in constant $O(1)$ memory.

**Q2: How do you detect Concept Drift using Prometheus metrics?**
> **Answer:** By exposing prediction class distributions (or continuous prediction score histograms) as Prometheus metrics and alerting in Grafana when the positive prediction rate deviates significantly from baseline.